# 01 — Your first oriented detections

Download a pretrained detector from the Hugging Face Hub, run it on a satellite tile from Python, and read what comes out. By the end you know what an oriented bounding box is, why overhead imagery needs one, and where a checkpoint's deploy settings live.

| | |
|---|---|
| Needs | Internet (one ≈330 MB checkpoint download) |
| Kaggle | Yes — no dataset input, Internet ON |
| GPU | Optional; CPU takes a few seconds per 1024 px tile |
| Posts | [Oriented-Det v0.1.0](https://deeplearning.earth/posts/2026-06-22_oriented-det_v0_1_0_sovereign_oriented_object_detection_for_eo/) · [Oriented object detection on macOS, in pure Python](https://deeplearning.earth/posts/2026-06-25_oriented_object_detection_on_macos_in_pure_python/) |

## Why oriented boxes?

Most detectors describe an object with an **axis-aligned box** (AABB): `x1, y1, x2, y2`, edges parallel to the image borders. That works for street photos, where cars and people are mostly upright. Seen from above there is no "up": ships, planes, and trucks point in every direction. The AABB of a ship lying at 45° is mostly water, and the AABBs of cars parked diagonally overlap each other so much that duplicate removal (NMS) starts deleting real cars.

An **oriented bounding box** (OBB) adds one number, the rotation: `cx, cy, w, h, θ`. The box hugs the object, neighbours stop overlapping, and you get heading, length, and width for free — the quantities an analyst actually wants.

OrientedDet is built around this five-number box (`RBox`) end to end: labels, augmentation, losses, NMS, and evaluation are all rotation-aware and written in plain PyTorch (no custom CUDA), so the same code runs on CPU, Apple MPS, and NVIDIA GPUs.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd))
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Pick a checkpoint

Pretrained weights live on the Hub under short **slugs**: `<detector>_<dataset>_<angle convention>_<schedule>`. `oriented_rcnn_dota_le90_1x` is Oriented R-CNN trained on DOTA v1.0 for 12 epochs ("1×") with the `le90` angle convention (lesson 02). It scores **76.73%** mAP50 on the official DOTA Task 1 test server.

In [ ]:
from oriented_det.pretrained import list_assets

for slug in list_assets():
    print(slug)

## 2. Download the weights and their config

Every Hub checkpoint has a **sidecar JSON** next to it: the exact training recipe plus a `production` block of deploy settings. The model is rebuilt from that config, then the weights are loaded into it. Three calls:

1. `ensure_checkpoint("hf://<slug>")` downloads the `.pth` once and returns its local path.
2. `hf_hub_download(...)` fetches the sidecar `.json` next to it (skipped when it is already there).
3. `load_model_from_checkpoint(ckpt, config)` builds the network and returns `(model, config, class_names)`.

In [ ]:
from huggingface_hub import hf_hub_download
from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config
from oriented_det.runtime.checkpoint import load_model_from_checkpoint

SLUG = "oriented_rcnn_dota_le90_1x"
ckpt = ensure_checkpoint(f"hf://{SLUG}")
manifest = load_manifest()
sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
    manifest["repo_id"], ckpt.with_suffix(".json").name,
    revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
model, config, class_names = load_model_from_checkpoint(str(ckpt), str(sidecar), device=DEVICE)

print(ckpt.name, f"{ckpt.stat().st_size / 1e6:.0f} MB")
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")
print(len(class_names), "classes:", ", ".join(class_names))

The `production` block is what makes a checkpoint deployable without guesswork. Its score floor was picked on validation data (best F1 minus a 0.05 margin), and the preprocessing tells inference how to resize and normalise pixels exactly as in training.

In [ ]:
from oriented_det.train.config import get_preprocessing_params

p = config.production
print(f"score floor {p.score_threshold}   NMS IoU {p.final_nms_iou_threshold}   window overlap {p.overlap_pixels} px")
pre = get_preprocessing_params(config)
print({k: pre[k] for k in ("resize_mode", "target_size", "normalize_mean", "normalize_std") if k in pre})

## 3. Run it

`run_inference_auto` preprocesses the image, runs the network, and maps boxes back to original pixels. Images larger than the 1024 px training canvas are cut into overlapping windows automatically (lesson 06). The demo tile is exactly 1024 px, so this is a single forward pass.

In [ ]:
from PIL import Image
from oriented_det.runtime.inference import run_inference_auto
from oriented_det.train.config import resolve_inference_score_threshold

image = Image.open(demo_file("demo.jpg")).convert("RGB")
t0 = time.perf_counter()
dets = run_inference_auto(
    image=image, model=model, device=DEVICE,
    preprocessing=get_preprocessing_params(config),
    score_threshold=resolve_inference_score_threshold(config),
    nms_threshold=config.production.final_nms_iou_threshold,
    class_names=class_names,
)
print(f"{len(dets)} detections on a {image.width}x{image.height} image in {time.perf_counter() - t0:.1f} s")

## 4. Read a detection

Each detection is a dict with an `RBox` (centre, width, height, angle in radians), a confidence `score`, and a 1-based class `label`.

In [ ]:
import math
from collections import Counter

d = max(dets, key=lambda d: d["score"])
r = d["rbox"]
print(f"{class_names[d['label'] - 1]}  score={d['score']:.2f}")
print(f"centre=({r.cx:.0f}, {r.cy:.0f})  size={r.width:.0f}x{r.height:.0f} px  angle={math.degrees(r.angle):.0f} deg")
print(Counter(class_names[d["label"] - 1] for d in dets).most_common())

## 5. Oriented vs axis-aligned, measured

Draw the same detections both ways and count what each representation claims. `axis_aligned_bounds()` is the AABB enclosing an oriented box.

In [ ]:
from oriented_det.ops import rbox_iou
from oriented_det.runtime.inference import visualize_results
from oriented_det.utils.viz import DrawingSpec, draw_polygons


def aabb_iou(a, b):
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)


boxes = [d["rbox"] for d in dets]
aabbs = [b.axis_aligned_bounds() for b in boxes]
pairs = [(i, j) for i in range(len(boxes)) for j in range(i + 1, len(boxes))]
area_ratio = sum((x2 - x1) * (y2 - y1) for x1, y1, x2, y2 in aabbs) / sum(b.area for b in boxes)
print(f"axis-aligned boxes cover {area_ratio:.1f}x the area of the oriented ones")
print("pairs overlapping at IoU > 0.1:",
      sum(rbox_iou(boxes[i], boxes[j]) > 0.1 for i, j in pairs), "oriented vs",
      sum(aabb_iou(aabbs[i], aabbs[j]) > 0.1 for i, j in pairs), "axis-aligned")

aabb_view = draw_polygons(image.copy(), [[(x1, y1), (x2, y1), (x2, y2), (x1, y2)] for x1, y1, x2, y2 in aabbs],
                          specs=[DrawingSpec(outline=(255, 64, 64), width=2)] * len(aabbs))
show(visualize_results(image.copy(), dets, class_names), aabb_view,
     titles=("oriented (what the model predicts)", "axis-aligned equivalent"))

The overlap count is the practical argument: every axis-aligned pair above the NMS threshold is a real object that an AABB detector risks deleting as a "duplicate".

## 6. The same thing from the command line

Everything above is also one CLI call. `odet image-demo` takes an image and a checkpoint (local path or `hf://slug`), finds the sidecar config next to the checkpoint, and writes an annotated PNG. `ODET` in the setup cell is `python -m oriented_det.cli`, the same as typing `odet` in a terminal.

In [ ]:
out = WORK / "01_demo_cli.png"
run([*ODET, "image-demo", demo_file("demo.jpg"), f"hf://{SLUG}", "--device", DEVICE, "--out-file", out])
show(Image.open(out), titles=("odet image-demo",))

## What to try next

- Pass `score_threshold=0.3` in section 3 and look at what appears: low-confidence detections are where a model's confusions live.
- Set `SLUG = "rotated_fcos_dota_le90_1x"` and rerun. Its deploy floor is very different; lesson 04 explains why.
- `hub_model(slug)` and `detect(...)` in the setup cell wrap sections 2–3. Later lessons use them so the interesting code stays short.
- **Next:** [02 — Boxes, angles, and rotated NMS](02_boxes_angles_nms.ipynb) opens up the `RBox` itself.